```python
# ============================================================
# FALL DETECTION TRAINING - FAST GRADIENT BOOSTING VERSION
# Tối ưu cho MPU6050 đeo trước ngực / server load nhanh
# Datasets hỗ trợ:
#   1) SisFall Kaggle: nvnikhil0001/sis-fall-original-dataset
#   2) KFall Kaggle hoặc thư mục KFall tự tải: usmanabbasi2002/kfall-dataset
# Output:
#   fall_gbdt_model.joblib  -> load nhanh hơn TensorFlow .keras
#   fall_metadata.json      -> thông tin feature + threshold
# ============================================================
```

In [ ]:
# Cell 1 - Cài thư viện
# Chạy 1 lần trong notebook/Colab/Jupyter
# %pip install -U kagglehub numpy pandas scikit-learn scipy joblib matplotlib openpyxl

In [ ]:
# Cell 2 - Import & config
import os
import re
import json
import time
import warnings
from pathlib import Path
from collections import deque

import kagglehub
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import skew, kurtosis
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")

# ====== CẤU HÌNH CHÍNH ======
WINDOW_SIZE = 150       # 150 mẫu ~ 1.5 giây nếu 100 Hz
STRIDE = 50             # overlap 66%
TARGET_HZ = 100         # đưa SisFall 200 Hz về gần 100 Hz để giống MPU6050 thực tế hơn
USE_KFALL = True        # nếu không tải được KFall, code vẫn train SisFall
USE_SISFALL = True

# Lưu model ở thư mục hiện tại hoặc đổi sang d:/Ky6/PBL
BASE_DIR = Path("./fall_model_output")
BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = BASE_DIR / "fall_gbdt_model.joblib"
METADATA_FILE = BASE_DIR / "fall_metadata.json"
REPORT_FILE = BASE_DIR / "fall_report.json"

RANDOM_STATE = 42

print("Save dir:", BASE_DIR.resolve())

In [ ]:
# Cell 3 - Tải dataset
def safe_download_kaggle(slug: str):
    try:
        p = kagglehub.dataset_download(slug)
        print(f"Downloaded {slug} -> {p}")
        return Path(p)
    except Exception as e:
        print(f"Không tải được {slug}: {e}")
        return None

sisfall_path = safe_download_kaggle("nvnikhil0001/sis-fall-original-dataset") if USE_SISFALL else None

# KFall chính thức cần xin quyền trên Google Site; bản Kaggle có thể thay đổi cấu trúc.
# Nếu Kaggle không tải được, đặt KFALL_LOCAL_PATH đến thư mục bạn tự tải.
KFALL_LOCAL_PATH = None  # ví dụ: Path(r"D:\datasets\KFall")
kfall_path = Path(KFALL_LOCAL_PATH) if KFALL_LOCAL_PATH else (
    safe_download_kaggle("usmanabbasi2002/kfall-dataset") if USE_KFALL else None
)

print("sisfall_path:", sisfall_path)
print("kfall_path:", kfall_path)

In [ ]:
# Cell 4 - Hàm đọc file và chuẩn hóa đơn vị
def parse_numeric_text_file(filepath: Path, min_cols=6):
    rows = []
    with open(filepath, "r", errors="ignore") as f:
        for line in f:
            line = line.strip().replace(";", ",")
            if not line:
                continue
            parts = re.split(r"[,\s]+", line)
            vals = []
            for x in parts:
                try:
                    vals.append(float(x))
                except Exception:
                    pass
            if len(vals) >= min_cols:
                rows.append(vals)
    if not rows:
        return np.empty((0, min_cols), dtype=np.float32)

    max_len = max(len(r) for r in rows)
    arr = np.full((len(rows), max_len), np.nan, dtype=np.float32)
    for i, r in enumerate(rows):
        arr[i, :len(r)] = r
    return arr


def convert_sisfall_units(raw):
    """
    SisFall thường lưu raw sensor:
    - ADXL345 ±16g, 13-bit: g ≈ raw / 256
    - ITG3200 ±2000 dps, 16-bit: dps ≈ raw / 16.384
    Dùng 6 cột đầu: ax,ay,az,gx,gy,gz theo notebook gốc của bạn.
    Nếu dataset Kaggle đã scale sẵn, hàm auto kiểm tra biên độ để tránh scale 2 lần.
    """
    raw = raw[:, :6].astype(np.float32)

    acc = raw[:, :3].copy()
    gyro = raw[:, 3:6].copy()

    # Nếu acc median quá lớn, khả năng là raw count -> đổi sang g
    med_acc = np.nanmedian(np.sqrt(np.sum(acc**2, axis=1)))
    if med_acc > 20:
        acc = acc / 256.0

    # Nếu gyro quá lớn, khả năng là raw count -> đổi sang deg/s
    med_gyro_abs = np.nanmedian(np.abs(gyro))
    if med_gyro_abs > 500:
        gyro = gyro / 16.384

    return np.hstack([acc, gyro]).astype(np.float32)


def read_sisfall_file(filepath: Path):
    raw = parse_numeric_text_file(filepath, min_cols=6)
    if raw.shape[0] == 0:
        return raw[:, :6].astype(np.float32)
    data = convert_sisfall_units(raw)

    # SisFall gốc 200 Hz; giảm còn xấp xỉ 100 Hz để giống MPU6050 server/ESP32 hơn
    if TARGET_HZ == 100:
        data = data[::2]

    return data.astype(np.float32)


def infer_kfall_columns(df: pd.DataFrame):
    """
    KFall CSV chính thức có 11 cột: timestamp, frame, acc 3 trục, gyro 3 trục, euler 3 trục.
    Hàm này cố gắng nhận diện tên cột linh hoạt.
    """
    cols = list(df.columns)
    low = {c: str(c).lower().strip() for c in cols}

    def find_one(candidates, fallback_idx=None):
        for cand in candidates:
            cand = cand.lower()
            for c in cols:
                name = low[c]
                if cand == name or cand in name:
                    return c
        if fallback_idx is not None and fallback_idx < len(cols):
            return cols[fallback_idx]
        return None

    # fallback theo thứ tự KFall chính thức: timestamp, frame, accX, accY, accZ, gyroX, gyroY, gyroZ, eulerX...
    ax = find_one(["accx", "acc_x", "acceleration x", "acc x", "ax"], 2)
    ay = find_one(["accy", "acc_y", "acceleration y", "acc y", "ay"], 3)
    az = find_one(["accz", "acc_z", "acceleration z", "acc z", "az"], 4)
    gx = find_one(["gyrox", "gyro_x", "angular velocity x", "gyrx", "gx"], 5)
    gy = find_one(["gyroy", "gyro_y", "angular velocity y", "gyry", "gy"], 6)
    gz = find_one(["gyroz", "gyro_z", "angular velocity z", "gyrz", "gz"], 7)

    return [ax, ay, az, gx, gy, gz]


def read_kfall_csv(filepath: Path):
    try:
        df = pd.read_csv(filepath)
    except Exception:
        df = pd.read_csv(filepath, header=None)

    cols = infer_kfall_columns(df)
    data = df[cols].apply(pd.to_numeric, errors="coerce").dropna().to_numpy(dtype=np.float32)

    # KFall chính thức: acceleration đơn vị g, gyro deg/s.
    # Nếu một bản Kaggle chuyển sang m/s^2, med_acc ≈ 9.81 thì đổi về g.
    if len(data) > 0:
        med_acc = np.nanmedian(np.sqrt(np.sum(data[:, :3] ** 2, axis=1)))
        if 6.0 < med_acc < 14.0:
            data[:, :3] /= 9.80665

    return data.astype(np.float32)

In [ ]:
# Cell 5 - Feature engineering cho Gradient Boosting
def safe_stat(x):
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return [0.0] * 8
    return [
        float(np.mean(x)),
        float(np.std(x)),
        float(np.min(x)),
        float(np.max(x)),
        float(np.ptp(x)),
        float(np.median(x)),
        float(skew(x, bias=False)) if len(x) > 2 else 0.0,
        float(kurtosis(x, bias=False)) if len(x) > 3 else 0.0,
    ]


def raw6_to_time_features(raw6):
    """
    Input raw6 shape (N,6): ax,ay,az in g; gx,gy,gz in deg/s
    Output time features shape (N, 11)
    """
    raw6 = np.asarray(raw6, dtype=np.float32)
    ax, ay, az, gx, gy, gz = raw6.T

    acc_mag = np.sqrt(ax**2 + ay**2 + az**2)
    gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)
    jerk = np.abs(np.diff(acc_mag, prepend=acc_mag[0]))
    roll = np.arctan2(ay, az + 1e-6)
    pitch = np.arctan2(-ax, np.sqrt(ay**2 + az**2) + 1e-6)

    return np.column_stack([
        ax, ay, az, gx, gy, gz,
        acc_mag, gyro_mag, jerk, roll, pitch
    ]).astype(np.float32)


TIME_FEATURE_KEYS = [
    "ax", "ay", "az", "gx", "gy", "gz",
    "acc_mag", "gyro_mag", "jerk", "roll", "pitch"
]


def extract_window_features(raw6_window):
    """
    Biến 150x6 thành vector thống kê nhỏ -> Gradient Boosting load/predict rất nhanh.
    Feature tập trung vào: impact, free-fall, jerk, gyro spike, đổi tư thế, bất động sau va chạm.
    """
    tfm = raw6_to_time_features(raw6_window)
    d = {}

    for i, name in enumerate(TIME_FEATURE_KEYS):
        vals = safe_stat(tfm[:, i])
        keys = ["mean", "std", "min", "max", "range", "median", "skew", "kurt"]
        for k, v in zip(keys, vals):
            d[f"{name}_{k}"] = v

    acc = tfm[:, TIME_FEATURE_KEYS.index("acc_mag")]
    gyro = tfm[:, TIME_FEATURE_KEYS.index("gyro_mag")]
    jerk = tfm[:, TIME_FEATURE_KEYS.index("jerk")]
    roll = tfm[:, TIME_FEATURE_KEYS.index("roll")]
    pitch = tfm[:, TIME_FEATURE_KEYS.index("pitch")]

    # Feature vật lý chính
    d["acc_peak"] = float(np.max(acc))
    d["acc_min"] = float(np.min(acc))
    d["freefall_depth"] = float(max(0.0, 1.0 - np.min(acc)))
    d["impact_minus_min"] = float(np.max(acc) - np.min(acc))
    d["jerk_peak"] = float(np.max(jerk))
    d["gyro_peak"] = float(np.max(gyro))

    # Đổi tư thế theo độ
    d["roll_change_deg"] = float(np.ptp(roll) * 180 / np.pi)
    d["pitch_change_deg"] = float(np.ptp(pitch) * 180 / np.pi)
    d["angle_change_deg"] = float(max(d["roll_change_deg"], d["pitch_change_deg"]))
    d["roll_final_deg"] = float(roll[-1] * 180 / np.pi)
    d["pitch_final_deg"] = float(pitch[-1] * 180 / np.pi)

    # Bất động sau impact: lấy 1/3 cuối window
    tail = max(10, len(acc) // 3)
    d["post_acc_std"] = float(np.std(acc[-tail:]))
    d["post_gyro_std"] = float(np.std(gyro[-tail:]))
    d["post_motion_score"] = float(d["post_acc_std"] + 0.01 * d["post_gyro_std"])

    # Tương quan giữa acc & gyro: nhiều cú fall có spike cùng lúc
    if np.std(acc) > 1e-6 and np.std(gyro) > 1e-6:
        d["acc_gyro_corr"] = float(np.corrcoef(acc, gyro)[0, 1])
    else:
        d["acc_gyro_corr"] = 0.0

    return d


def make_windows_from_raw6(raw6, label, group_id, stride=STRIDE):
    rows, labels, groups = [], [], []
    n = len(raw6)
    if n < WINDOW_SIZE:
        return rows, labels, groups

    for start in range(0, n - WINDOW_SIZE + 1, stride):
        win = raw6[start:start + WINDOW_SIZE]
        rows.append(extract_window_features(win))
        labels.append(int(label))
        groups.append(group_id)
    return rows, labels, groups

In [ ]:
# Cell 6 - Load SisFall
def load_sisfall_dataset(root: Path):
    rows, labels, groups = [], [], []
    if root is None or not Path(root).exists():
        return rows, labels, groups

    fall_files = normal_files = used = 0
    for dirpath, _, files in os.walk(root):
        for fn in files:
            if not fn.lower().endswith((".txt", ".csv")):
                continue

            # SisFall: F... = fall, D... = daily/normal
            if fn.startswith("F"):
                label = 1
                fall_files += 1
            elif fn.startswith("D"):
                label = 0
                normal_files += 1
            else:
                continue

            fp = Path(dirpath) / fn
            raw6 = read_sisfall_file(fp)
            if len(raw6) < WINDOW_SIZE:
                continue

            # group theo subject/file để tránh leak train-test
            group_id = f"sisfall::{Path(dirpath).name}::{fn}"
            r, y, g = make_windows_from_raw6(raw6, label, group_id)
            rows.extend(r); labels.extend(y); groups.extend(g)
            used += 1

    print(f"SisFall used files={used}, fall_files={fall_files}, normal_files={normal_files}, windows={len(rows)}")
    return rows, labels, groups

In [ ]:
# Cell 7 - Load KFall
def parse_kfall_name(filepath: Path):
    """
    Ví dụ chính thức: SA06T20R01.csv
    return: subject='SA06', task=20, trial=1
    """
    m = re.search(r"(SA\d+).*?T(\d+).*?R(\d+)", filepath.stem, flags=re.I)
    if not m:
        return None, None, None
    return m.group(1).upper(), int(m.group(2)), int(m.group(3))


def load_kfall_labels(root: Path):
    """
    Đọc label_data xlsx/csv nếu có.
    Trả về dict key=(subject,task,trial) -> (onset_frame, impact_frame)
    """
    labels = {}
    if root is None or not Path(root).exists():
        return labels

    files = []
    for ext in ("*.xlsx", "*.xls", "*.csv"):
        files.extend(Path(root).rglob(ext))

    for fp in files:
        # chỉ ưu tiên file có chữ label hoặc nằm trong label_data
        if "label" not in str(fp).lower():
            continue
        subject_match = re.search(r"(SA\d+)", fp.stem, flags=re.I)
        subject_from_file = subject_match.group(1).upper() if subject_match else None

        try:
            df = pd.read_excel(fp) if fp.suffix.lower() in [".xlsx", ".xls"] else pd.read_csv(fp)
        except Exception:
            continue

        low = {c: str(c).lower() for c in df.columns}

        def col_has(words):
            for c, name in low.items():
                if all(w in name for w in words):
                    return c
            return None

        task_col = col_has(["task"]) or df.columns[0]
        trial_col = col_has(["trial"]) or df.columns[2 if len(df.columns) > 2 else 0]
        onset_col = col_has(["onset"]) or col_has(["fall", "start"])
        impact_col = col_has(["impact"])

        for _, row in df.iterrows():
            try:
                subject = subject_from_file
                if subject is None:
                    for val in row.values:
                        mm = re.search(r"(SA\d+)", str(val), flags=re.I)
                        if mm:
                            subject = mm.group(1).upper()
                            break
                if subject is None:
                    continue

                task_raw = str(row[task_col])
                mt = re.search(r"(\d+)", task_raw)
                if not mt:
                    continue
                task = int(mt.group(1))
                trial = int(re.search(r"(\d+)", str(row[trial_col])).group(1))

                onset = int(float(row[onset_col])) if onset_col is not None and pd.notna(row[onset_col]) else None
                impact = int(float(row[impact_col])) if impact_col is not None and pd.notna(row[impact_col]) else onset

                if onset is not None or impact is not None:
                    labels[(subject, task, trial)] = (onset, impact)
            except Exception:
                continue

    print("KFall label rows:", len(labels))
    return labels


def make_kfall_windows(raw6, group_id, fall_interval=None):
    """
    Nếu có label fall onset/impact: chỉ gán fall cho window bao quanh sự kiện.
    Window ngoài sự kiện trong file fall được bỏ qua hoặc gán normal? Ở đây bỏ qua để tránh nhiễu nhãn.
    """
    rows, labels, groups = [], [], []
    n = len(raw6)
    if n < WINDOW_SIZE:
        return rows, labels, groups

    for start in range(0, n - WINDOW_SIZE + 1, STRIDE):
        end = start + WINDOW_SIZE
        if fall_interval is None:
            label = 0
        else:
            onset, impact = fall_interval
            center = impact if impact is not None else onset
            # vùng fall: từ trước onset 0.5s đến sau impact 0.5s
            left = max(0, (onset if onset is not None else center) - 50)
            right = min(n, (impact if impact is not None else center) + 50)
            overlap = max(0, min(end, right) - max(start, left))
            if overlap < 20:
                continue
            label = 1

        rows.append(extract_window_features(raw6[start:end]))
        labels.append(label)
        groups.append(group_id)

    return rows, labels, groups


def load_kfall_dataset(root: Path):
    rows, labels, groups = [], [], []
    if root is None or not Path(root).exists():
        return rows, labels, groups

    kfall_labels = load_kfall_labels(root)

    csv_files = list(Path(root).rglob("*.csv"))
    used = fall_used = adl_used = 0

    for fp in csv_files:
        # bỏ file label csv
        if "label" in str(fp).lower():
            continue

        subject, task, trial = parse_kfall_name(fp)
        if subject is None:
            continue

        raw6 = read_kfall_csv(fp)
        if len(raw6) < WINDOW_SIZE:
            continue

        key = (subject, task, trial)
        fall_interval = kfall_labels.get(key, None)

        # Nếu không có labels thì bỏ qua file fall không rõ nhãn, chỉ dùng file không có label làm normal.
        # Nếu dataset Kaggle có label khác, hãy kiểm tra cấu trúc và cập nhật load_kfall_labels.
        group_id = f"kfall::{subject}::T{task:02d}::R{trial:02d}"

        if fall_interval is not None:
            r, y, g = make_kfall_windows(raw6, group_id, fall_interval=fall_interval)
            fall_used += 1
        else:
            r, y, g = make_windows_from_raw6(raw6, 0, group_id)
            adl_used += 1

        rows.extend(r); labels.extend(y); groups.extend(g)
        used += 1

    print(f"KFall used files={used}, fall_files_with_label={fall_used}, adl_or_unlabeled_files={adl_used}, windows={len(rows)}")
    return rows, labels, groups

In [ ]:
# Cell 8 - Gom dữ liệu
all_rows, all_labels, all_groups = [], [], []

sis_rows, sis_y, sis_g = load_sisfall_dataset(sisfall_path)
all_rows += sis_rows; all_labels += sis_y; all_groups += sis_g

k_rows, k_y, k_g = load_kfall_dataset(kfall_path)
all_rows += k_rows; all_labels += k_y; all_groups += k_g

df = pd.DataFrame(all_rows)
y = np.asarray(all_labels, dtype=np.int32)
groups = np.asarray(all_groups)

print("Total windows:", len(df))
print("Feature dim:", df.shape[1] if len(df) else 0)
print("Fall windows:", int(np.sum(y == 1)), "Normal windows:", int(np.sum(y == 0)))

if len(df) == 0:
    raise RuntimeError("Không load được dataset nào. Kiểm tra Kaggle login/path dataset.")

if np.sum(y == 1) == 0 or np.sum(y == 0) == 0:
    raise RuntimeError("Dataset chỉ có 1 lớp. Cần cả FALL và NORMAL.")

feature_names = list(df.columns)

In [ ]:
# Cell 9 - Split theo group/file để tránh data leakage
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(df, y, groups=groups))

X_train = df.iloc[train_idx].copy()
y_train = y[train_idx]
groups_train = groups[train_idx]

X_temp = df.iloc[temp_idx].copy()
y_temp = y[temp_idx]
groups_temp = groups[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_rel_idx, test_rel_idx = next(gss2.split(X_temp, y_temp, groups=groups_temp))

X_val = X_temp.iloc[val_rel_idx].copy()
y_val = y_temp[val_rel_idx]
X_test = X_temp.iloc[test_rel_idx].copy()
y_test = y_temp[test_rel_idx]

print("Train:", X_train.shape, "fall:", int(np.sum(y_train==1)), "normal:", int(np.sum(y_train==0)))
print("Val:  ", X_val.shape, "fall:", int(np.sum(y_val==1)), "normal:", int(np.sum(y_val==0)))
print("Test: ", X_test.shape, "fall:", int(np.sum(y_test==1)), "normal:", int(np.sum(y_test==0)))

In [ ]:
# Cell 10 - Train HistGradientBoostingClassifier
# Lý do chọn HGBDT:
# - Không cần TensorFlow -> load nhanh hơn nhiều
# - Học tốt feature phi tuyến: impact + jerk + gyro + orientation + immobility
# - Thích hợp dữ liệu tabular/statistical features
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gbdt", HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.06,
        max_iter=220,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.05,
        early_stopping=True,
        validation_fraction=0.12,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
    ))
])

t0 = time.time()
model.fit(X_train, y_train, gbdt__sample_weight=sample_weight)
print(f"Training time: {time.time() - t0:.2f}s")

In [ ]:
# Cell 11 - Tối ưu threshold trên validation
val_prob = model.predict_proba(X_val)[:, 1]
precision_arr, recall_arr, thresholds = precision_recall_curve(y_val, val_prob)

f1_scores = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-8)
best_idx = int(np.argmax(f1_scores[:-1]))
best_threshold = float(thresholds[best_idx])

print("Best threshold:", best_threshold)
print("Val precision:", float(precision_arr[best_idx]))
print("Val recall:", float(recall_arr[best_idx]))
print("Val F1:", float(f1_scores[best_idx]))

# Có thể chọn threshold bảo thủ hơn nếu muốn giảm false alarm:
# best_threshold = max(best_threshold, 0.65)

In [ ]:
# Cell 12 - Đánh giá test
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(np.int32)

print(classification_report(y_test, test_pred, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

metrics = {
    "threshold": best_threshold,
    "roc_auc": float(roc_auc_score(y_test, test_prob)) if len(np.unique(y_test)) == 2 else None,
    "average_precision": float(average_precision_score(y_test, test_prob)) if len(np.unique(y_test)) == 2 else None,
    "n_train": int(len(X_train)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
    "n_features": int(len(feature_names)),
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "target_hz": TARGET_HZ,
}

print(metrics)

In [ ]:
# Cell 13 - Lưu model
bundle = {
    "model": model,
    "feature_names": feature_names,
    "threshold": best_threshold,
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "target_hz": TARGET_HZ,
    "time_feature_keys": TIME_FEATURE_KEYS,
}

joblib.dump(bundle, MODEL_FILE, compress=3)

metadata = {
    "feature_names": feature_names,
    "threshold": best_threshold,
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "target_hz": TARGET_HZ,
    "model_type": "sklearn HistGradientBoostingClassifier",
    "notes": "Input raw6: ax,ay,az in g; gx,gy,gz in deg/s. MPU6050 đeo trước ngực cần tự thu thêm dữ liệu thật để fine-tune threshold.",
}
METADATA_FILE.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
REPORT_FILE.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", MODEL_FILE.resolve())
print("Saved:", METADATA_FILE.resolve())
print("Saved:", REPORT_FILE.resolve())

In [ ]:
# Cell 14 - Test tốc độ load và predict
t0 = time.time()
prod = joblib.load(MODEL_FILE)
load_ms = (time.time() - t0) * 1000

sample = X_test.iloc[[0]]

# warm-up
_ = prod["model"].predict_proba(sample)[:, 1]

N_RUN = 1000
t0 = time.time()
for _ in range(N_RUN):
    prob = float(prod["model"].predict_proba(sample)[:, 1][0])
pred_ms = (time.time() - t0) / N_RUN * 1000

print(f"Load time: {load_ms:.2f} ms")
print(f"Average predict time: {pred_ms:.4f} ms")
print("Sample prob:", prob)

In [ ]:
# Cell 15 - Hàm predict thật trên server từ buffer MPU6050 trước ngực
def physical_gate_from_raw6(raw6_window):
    """
    Gate mềm để giảm false alarm và tránh chạy model khi quá chắc chắn là bình thường.
    Vì đeo trước ngực khác waist/low-back, các ngưỡng dưới nên hiệu chỉnh bằng dữ liệu tự thu.
    """
    tfm = raw6_to_time_features(raw6_window)
    acc = tfm[:, TIME_FEATURE_KEYS.index("acc_mag")]
    gyro = tfm[:, TIME_FEATURE_KEYS.index("gyro_mag")]
    jerk = tfm[:, TIME_FEATURE_KEYS.index("jerk")]
    roll = tfm[:, TIME_FEATURE_KEYS.index("roll")]
    pitch = tfm[:, TIME_FEATURE_KEYS.index("pitch")]

    acc_peak = float(np.max(acc))
    acc_min = float(np.min(acc))
    jerk_peak = float(np.max(jerk))
    gyro_peak = float(np.max(gyro))
    angle_change_deg = float(max(np.ptp(roll), np.ptp(pitch)) * 180 / np.pi)
    post_acc_std = float(np.std(acc[-max(10, len(acc)//3):]))

    info = {
        "acc_peak": round(acc_peak, 4),
        "acc_min": round(acc_min, 4),
        "jerk_peak": round(jerk_peak, 4),
        "gyro_peak": round(gyro_peak, 4),
        "angle_change_deg": round(angle_change_deg, 2),
        "post_acc_std": round(post_acc_std, 4),
    }

    # Gate không quá cứng: chest có thể impact thấp hơn waist
    suspicious = (
        (acc_peak >= 1.6 and jerk_peak >= 0.25) or
        (gyro_peak >= 180 and angle_change_deg >= 25) or
        (acc_min <= 0.55 and acc_peak >= 1.4)
    )

    return suspicious, info


def predict_raw6_window(raw6_window, loaded_bundle=None):
    """
    raw6_window: numpy array shape (150, 6)
                 ax,ay,az đơn vị g; gx,gy,gz đơn vị deg/s
    return dict status/probability/gate
    """
    if loaded_bundle is None:
        loaded_bundle = joblib.load(MODEL_FILE)

    suspicious, gate = physical_gate_from_raw6(raw6_window)

    # Nếu chắc chắn không có impact/rotation đáng kể -> normal ngay
    if not suspicious:
        return {
            "status": "NORMAL",
            "pred": 0,
            "fall_probability": 0.0,
            "threshold": loaded_bundle["threshold"],
            "gate": {**gate, "reason": "not_suspicious_physics"},
        }

    feats = extract_window_features(raw6_window)
    X_one = pd.DataFrame([feats], columns=loaded_bundle["feature_names"])
    prob = float(loaded_bundle["model"].predict_proba(X_one)[:, 1][0])
    pred = int(prob >= loaded_bundle["threshold"])

    return {
        "status": "FALL_ALERT" if pred else "NORMAL",
        "pred": pred,
        "fall_probability": round(prob, 4),
        "threshold": round(float(loaded_bundle["threshold"]), 4),
        "gate": {**gate, "reason": "run_model"},
    }


# Test bằng 1 window từ dataset test nếu còn dữ liệu gốc thì bạn thay bằng buffer thực tế
print("Predict function ready.")

In [ ]:
# Cell 16 - Ví dụ tích hợp MQTT/Serial: buffer 150 mẫu
# Mỗi sample gửi từ ESP32 nên là:
# {"ax":..., "ay":..., "az":..., "gx":..., "gy":..., "gz":...}
# ax/ay/az nên là g, gx/gy/gz nên là deg/s.
buffer = deque(maxlen=WINDOW_SIZE)

def add_sample_and_predict(sample_json, loaded_bundle=None):
    ax = float(sample_json.get("ax", 0.0))
    ay = float(sample_json.get("ay", 0.0))
    az = float(sample_json.get("az", 1.0))
    gx = float(sample_json.get("gx", 0.0))
    gy = float(sample_json.get("gy", 0.0))
    gz = float(sample_json.get("gz", 0.0))

    buffer.append([ax, ay, az, gx, gy, gz])

    if len(buffer) < WINDOW_SIZE:
        return None

    raw6_window = np.asarray(buffer, dtype=np.float32)
    return predict_raw6_window(raw6_window, loaded_bundle=loaded_bundle)

# loaded = joblib.load(MODEL_FILE)
# result = add_sample_and_predict({"ax":0.1,"ay":0.2,"az":1.0,"gx":0,"gy":0,"gz":0}, loaded)
# print(result)